# Ropedia Academy — B2 · Multi-view geometry & triangulation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B2.ipynb)

> **Triangulates a 3D point from two views via DLT and draws the camera centers, viewing rays, and their recovered intersection in 3D.**
>
> 用 DLT 从两个视图三角测量一个 3D 点，并在 3D 中画出相机中心、视线及其恢复出的交点。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B2

In [ ]:
import numpy as np, matplotlib.pyplot as plt

# Two calibrated views observe one 3D point; recover it by triangulation.
K = np.array([[800,0,320],[0,800,240],[0,0,1.]])
cam = lambda R, t: K @ np.hstack([R, t.reshape(3,1)])
P1 = cam(np.eye(3), np.zeros(3))
a = 0.4; R2 = np.array([[np.cos(a),0,np.sin(a)],[0,1,0],[-np.sin(a),0,np.cos(a)]])
C2 = np.array([1.5, 0, 0.]); P2 = cam(R2, -R2 @ C2)
X = np.array([0.5, -0.2, 4.0, 1.0])
proj = lambda P, X: (P @ X)[:2] / (P @ X)[2]

def triangulate(P1, P2, x1, x2):                  # DLT least squares
    A = np.stack([x1[0]*P1[2]-P1[0], x1[1]*P1[2]-P1[1], x2[0]*P2[2]-P2[0], x2[1]*P2[2]-P2[1]])
    _, _, Vt = np.linalg.svd(A); Xe = Vt[-1]; return Xe/Xe[3]
X_est = triangulate(P1, P2, proj(P1,X), proj(P2,X))
print("triangulation error:", round(np.linalg.norm(X_est - X), 5))

# Visualize: two camera centers, their rays, and the recovered intersection point.
fig = plt.figure(figsize=(5,4)); ax = fig.add_subplot(projection='3d')
for C in (np.zeros(3), C2):
    ax.plot(*np.stack([C, X[:3]]).T, 'k--', alpha=0.5); ax.scatter(*C, c='b', s=40)
ax.scatter(*X[:3], c='g', s=90, label='true'); ax.scatter(*X_est[:3], c='r', marker='x', s=90, label='triangulated')
ax.legend(); plt.title("two rays intersect at the 3D point"); plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B2
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks